# Modelflow live notebook using your local installation

This notebook is designed for a machine where **Modelflow is already installed locally**.

It uses the **real local modules**, not a mock parser:

- `modelflow.modelpattern`
- `modelflow.modelclass`

The goals are to:

1. inspect the actual local implementation of `nterm` and `model_parse`
2. run `model_parse` on live `FRML ... $` examples
3. inspect `BaseModel.analyzemodelnew`
4. try building a small model instance and expose the resulting structure

Because Modelflow versions can differ a bit, the notebook uses a few defensive checks and introspection helpers.


## 1. Import the local Modelflow package

If the import fails, set `REPO_ROOT` to the directory that contains the `modelflow` package.


In [2]:
from pathlib import Path
import sys
import inspect
from pprint import pprint

# If Modelflow is already installed, leave this as None.
# Otherwise set it to the folder that contains the `modelflow` package.
REPO_ROOT = None
# Example:
# REPO_ROOT = Path(r"/path/to/Modelflow2")

if REPO_ROOT is not None:
    REPO_ROOT = Path(REPO_ROOT)
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

import modelpattern as pt
import modelclass as mc

print("modelpattern.py:", pt.__file__)
print("modelclass.py  :", mc.__file__)


modelpattern.py: C:\modelflow2\modelflow\modelpattern.py
modelclass.py  : C:\modelflow2\modelflow\modelclass.py


## 2. Inspect the real token structure

The notebook below confirms the live local definition of `nterm`.


In [3]:
print("nterm object:")
print(pt.nterm)

print("\nSource line for nterm, if available:")
src = inspect.getsource(pt)
for line in src.splitlines():
    if "namedtuple('nterm'" in line or 'namedtuple("nterm"' in line:
        print(line)


nterm object:
<class 'modelpattern.nterm'>

Source line for nterm, if available:
nterm = namedtuple('nterm', ['number', 'op', 'var', 'lag'])
    # nterm = namedtuple('nterm', [ 'number', 'op', 'var', 'lag'])
    # nterm = namedtuple('nterm', [ 'number', 'op', 'var', 'lag'])
    #nterm = namedtuple('nterm', ['comment', 'number', 'op', 'var', 'lag'])
    #%%  nterm = namedtuple('nterm', ['number', 'op', 'var', 'lag'])


## 3. Inspect the real `model_parse` implementation

This cell prints the source of `model_parse` from your local installation.


In [4]:
print(inspect.getsource(pt.model_parse))


def model_parse(equations,funks=[]):
    '''Takes a model returns a list of tupels. Each tupel contains:
        :the compleete formular: 
        :FRML:
        :formular name:
        :the expression:
        :list of terms from the expression:  
            
     The purpose of this function is to make model analysis faster. this is 20 times faster than looping over espressions in a model
     
     This new model_parse handels lags of -0 or +0 which ocours in some models from world bank. 
   '''
    fatoms = namedtuple('fatoms', 'whole, frml ,frmlname, expression')
    # nterm = namedtuple('nterm', [ 'number', 'op', 'var', 'lag'])
    expressionre= udtrykre(funks)
    ibh = [(fatoms(*c),
            [ nterm(t[0],t[1],t[2], '' if t[3] == '-0' or t[3]=='+0' else t[3]) for t in expressionre.findall(c[3])])  
           for c in (split_frml(f)  for f in find_frml(equations.upper()) )]
    return ibh



## 4. Helper to display parsed formulas and tokens


In [5]:
def show_model_parse(equations, funks=None):
    if funks is None:
        parsed = pt.model_parse(equations)
    else:
        parsed = pt.model_parse(equations, funks=funks)

    for i, (fatoms, terms) in enumerate(parsed, start=1):
        print(f"\n=== Formula {i} ===")
        print("whole      :", fatoms.whole.strip())
        print("frml       :", fatoms.frml)
        print("frmlname   :", fatoms.frmlname)
        print("expression :", fatoms.expression.strip())
        print("\nTokens:")
        for j, t in enumerate(terms):
            print(f"{j:>3}: {t}")
    return parsed


## 5. Live example: tokenization with real local Modelflow


In [6]:
eq_text = '''
FRML <IDENT> A = B(-1) + 3.5 * MAX(C,D) $
FRML <IDENT> X = Y(+0) + Z(-0) + LOG(Q) $
'''

parsed = show_model_parse(eq_text)



=== Formula 1 ===
whole      : FRML <IDENT> A = B(-1) + 3.5 * MAX(C,D) $
frml       : FRML
frmlname   : <IDENT>
expression : A = B(-1) + 3.5 * MAX(C,D) $

Tokens:
  0: nterm(number='', op='', var='A', lag='')
  1: nterm(number='', op='=', var='', lag='')
  2: nterm(number='', op='', var='B', lag='-1')
  3: nterm(number='', op='+', var='', lag='')
  4: nterm(number='3.5', op='', var='', lag='')
  5: nterm(number='', op='*', var='', lag='')
  6: nterm(number='', op='MAX', var='', lag='')
  7: nterm(number='', op='(', var='', lag='')
  8: nterm(number='', op='', var='C', lag='')
  9: nterm(number='', op=',', var='', lag='')
 10: nterm(number='', op='', var='D', lag='')
 11: nterm(number='', op=')', var='', lag='')
 12: nterm(number='', op='$', var='', lag='')

=== Formula 2 ===
whole      : FRML <IDENT> X = Y(+0) + Z(-0) + LOG(Q) $
frml       : FRML
frmlname   : <IDENT>
expression : X = Y(+0) + Z(-0) + LOG(Q) $

Tokens:
  0: nterm(number='', op='', var='X', lag='')
  1: nterm(number='', 

Things to verify in the output:

- `B(-1)` should be one `nterm` with `var='B'` and `lag='-1'`
- `MAX` and `LOG` should appear in the `op` field
- `Y(+0)` and `Z(-0)` should normally be normalized to empty lag


## 6. Helper summaries over the real token stream


In [7]:
def classify_token(tok):
    if tok.number:
        return "number"
    if tok.op:
        return "operator/function"
    if tok.var:
        return "variable"
    return "unknown"

def token_table(terms):
    rows = []
    for i, t in enumerate(terms):
        rows.append({
            "pos": i,
            "kind": classify_token(t),
            "number": t.number,
            "op": t.op,
            "var": t.var,
            "lag": t.lag,
        })
    return rows

for fatoms, terms in parsed:
    print("\nExpression:", fatoms.expression.strip())
    pprint(token_table(terms))



Expression: A = B(-1) + 3.5 * MAX(C,D) $
[{'kind': 'variable', 'lag': '', 'number': '', 'op': '', 'pos': 0, 'var': 'A'},
 {'kind': 'operator/function',
  'lag': '',
  'number': '',
  'op': '=',
  'pos': 1,
  'var': ''},
 {'kind': 'variable',
  'lag': '-1',
  'number': '',
  'op': '',
  'pos': 2,
  'var': 'B'},
 {'kind': 'operator/function',
  'lag': '',
  'number': '',
  'op': '+',
  'pos': 3,
  'var': ''},
 {'kind': 'number', 'lag': '', 'number': '3.5', 'op': '', 'pos': 4, 'var': ''},
 {'kind': 'operator/function',
  'lag': '',
  'number': '',
  'op': '*',
  'pos': 5,
  'var': ''},
 {'kind': 'operator/function',
  'lag': '',
  'number': '',
  'op': 'MAX',
  'pos': 6,
  'var': ''},
 {'kind': 'operator/function',
  'lag': '',
  'number': '',
  'op': '(',
  'pos': 7,
  'var': ''},
 {'kind': 'variable', 'lag': '', 'number': '', 'op': '', 'pos': 8, 'var': 'C'},
 {'kind': 'operator/function',
  'lag': '',
  'number': '',
  'op': ',',
  'pos': 9,
  'var': ''},
 {'kind': 'variable', 'lag': '

## 7. Inspect the real `BaseModel.analyzemodelnew`


In [8]:
print(inspect.getsource(mc.BaseModel.analyzemodelnew))


    def analyzemodelnew(self, silent):
        ''' Analyze a model

        The function creats:**Self.allvar** is a dictory with an entry for every variable in the model 
        the key is the variable name. 
        For each endogeneous variable there is a directory with thees keys:

        :maxlag: The max lag for this variable
        :maxlead: The max Lead for this variable
        :endo: 1 if the variable is endogeneous (ie on the left hand side of =
        :frml: String with the formular for this variable
        :frmlnumber: The number of the formular 
        :varnr: Number of this variable 
        :terms: The frml for this variable translated to terms 
        :frmlname: The frmlname for this variable 
        :startnr: Start of this variable in gauss seidel solutio vector :Advanced:
        :matrix: This lhs element is a matrix
        :dropfrml: If this frml shoud be excluded from the evaluation.


        In addition theese properties will be created: 

        :endoge

## 8. Look at `BaseModel` and likely public model constructors

This is helpful because the exact public constructor can vary by version.


In [9]:
print("BaseModel signature:")
print(inspect.signature(mc.BaseModel))

candidate_names = [name for name in dir(mc) if name.lower() in {"model", "basemodel"}]
print("\nCandidate model classes/functions:")
for name in candidate_names:
    obj = getattr(mc, name)
    print(f"{name}: {obj}")
    try:
        print("  signature:", inspect.signature(obj))
    except Exception:
        pass


BaseModel signature:
(i_eq='', modelname='testmodel', silent=False, straight=False, funks=[], tabcomplete=True, previousbase=False, use_preorder=True, normalized=True, safeorder=False, var_description={}, model_description='', var_groups={}, reports={}, equations_latex='', eviews_dict={}, use_fbmin=False, substitution={}, **kwargs)

Candidate model classes/functions:
BaseModel: <class 'modelclass.BaseModel'>
  signature: (i_eq='', modelname='testmodel', silent=False, straight=False, funks=[], tabcomplete=True, previousbase=False, use_preorder=True, normalized=True, safeorder=False, var_description={}, model_description='', var_groups={}, reports={}, equations_latex='', eviews_dict={}, use_fbmin=False, substitution={}, **kwargs)
model: <class 'modelclass.model'>
  signature: (i_eq='', modelname='testmodel', silent=False, straight=False, funks=[], tabcomplete=True, previousbase=False, use_preorder=True, normalized=True, safeorder=False, var_description={}, model_description='', var_group

## 9. Try building a small live model instance

This cell tries a few common constructor patterns.  
It is intentionally defensive because local Modelflow variants can differ.


In [10]:
toy_model_text = '''
FRML <BEHAVIORAL> C   = A*YD + B*C(-1) $
FRML <IDENTITY>   YD  = W + TR $
FRML <IDENTITY>   GDP = C + I + G $
'''

def try_build_model(eq_text):
    attempts = []

    # Common candidates in likely order
    candidates = []
    for name in ["model", "BaseModel"]:
        if hasattr(mc, name):
            candidates.append((name, getattr(mc, name)))

    # Common call patterns
    call_patterns = [
        lambda cls: cls(eq_text),
        lambda cls: cls(eq_text, modelname="toy"),
        lambda cls: cls(i_eq=eq_text),
        lambda cls: cls(i_eq=eq_text, modelname="toy"),
    ]

    for name, cls in candidates:
        for k, make in enumerate(call_patterns, start=1):
            try:
                obj = make(cls)
                return obj, (name, k)
            except Exception as e:
                attempts.append((name, k, repr(e)))

    return None, attempts

model_obj, meta = try_build_model(toy_model_text)

if model_obj is None:
    print("Could not build a model automatically.")
    print("Attempts:")
    for item in meta:
        print(item)
else:
    print("Built model with:", meta)
    print("Type:", type(model_obj))
    print("Name:", getattr(model_obj, "name", None))


Built model with: ('model', 1)
Type: <class 'modelclass.model'>
Name: testmodel


## 10. Inspect key attributes after analysis

If model construction succeeded, this cell prints useful structural attributes.


In [11]:
if 'model_obj' in globals() and model_obj is not None:
    keys = sorted(k for k in vars(model_obj).keys() if not k.startswith("__"))
    print("Available attributes:")
    pprint(keys[:200])

    interesting = [
        "endogene", "exogene", "maxlag", "maxlead", "allvar", "solveorder",
        "preorder", "coreorder", "use_preorder", "strongblock", "topo",
        "graph", "endograph", "istopo", "equations", "name"
    ]

    print("\nSelected attributes:")
    for attr in interesting:
        if hasattr(model_obj, attr):
            print(f"\n--- {attr} ---")
            value = getattr(model_obj, attr)
            try:
                if isinstance(value, dict):
                    print("dict with keys:", list(value.keys())[:20])
                elif isinstance(value, (list, tuple, set)):
                    print(type(value), "len=", len(value))
                    print(list(value)[:20])
                else:
                    print(value)
            except Exception:
                print(repr(value))
else:
    print("No model object available.")


Available attributes:
['_endograph',
 '_substitution',
 '_use_preorder',
 '_var_description',
 'aequalterm',
 'allvar',
 'allvar_set',
 'blank',
 'endogene',
 'endogene_true',
 'equations',
 'equations_latex',
 'eviews_dict',
 'exogene',
 'exogene_true',
 'fix_add_factor',
 'fix_dummy',
 'fix_endo',
 'fix_value',
 'funks',
 'genrcolumns',
 'group_dict',
 'hybrid',
 'implicit',
 'istopo',
 'keep_solutions',
 'maxlag',
 'maxlead',
 'maxnavlen',
 'maxstart',
 'model_description',
 'name',
 'normalized',
 'nrorder',
 'previousbase',
 'reports',
 'safeorder',
 'save',
 'solveorder',
 'split_calc_add_factor',
 'straight',
 'tabcomplete',
 'topo',
 'use_fbmin',
 'v_nr',
 'var_groups']

Selected attributes:

--- endogene ---
<class 'set'> len= 3
['C', 'GDP', 'YD']

--- exogene ---
<class 'set'> len= 6
['A', 'TR', 'G', 'B', 'W', 'I']

--- maxlag ---
-1

--- maxlead ---
0

--- allvar ---
dict with keys: ['A', 'GDP', 'C', 'YD', 'TR', 'G', 'B', 'W', 'I']

--- solveorder ---
<class 'list'> len= 3
[

## 11. Show parsed terms stored inside the live model, if present

Different versions store tokenized equations under different attributes.  
This cell scans for likely places where the parsed token lists are stored.


In [12]:
if 'model_obj' in globals() and model_obj is not None:
    likely = [k for k in vars(model_obj).keys() if any(x in k.lower() for x in ["term", "equat", "var", "frml"])]
    print("Likely structural attributes:")
    pprint(likely)

    for attr in likely[:20]:
        value = getattr(model_obj, attr)
        if isinstance(value, dict) and value:
            print(f"\nAttribute {attr} is a dict; first keys:")
            print(list(value.keys())[:10])
            break
else:
    print("No model object available.")


Likely structural attributes:
['equations',
 'allvar',
 'aequalterm',
 'allvar_set',
 '_var_description',
 'var_groups',
 'equations_latex',
 '_prevar',
 '_epivar',
 '_corevar']

Attribute allvar is a dict; first keys:
['A', 'GDP', 'C', 'YD', 'TR', 'G', 'B', 'W', 'I']


## 12. Optional: inspect exact source for graph / ordering helpers

This helps connect `analyzemodelnew()` to dependency graph construction.


In [13]:
graph_related = [name for name in dir(mc.BaseModel) if any(x in name.lower() for x in ["topo", "graph", "block", "order"])]
print("Graph/order related BaseModel members:")
pprint(graph_related)

for name in graph_related[:10]:
    obj = getattr(mc.BaseModel, name)
    if callable(obj):
        print(f"\n===== {name} =====")
        try:
            print(inspect.getsource(obj))
        except Exception as e:
            print("Could not show source:", e)


Graph/order related BaseModel members:
['endograph']


## 13. A small manual checklist for your local run

When you run this notebook on your local Modelflow install, you can verify:

- the exact `nterm` definition in your installed `modelpattern.py`
- the exact regex/token behavior of `model_parse`
- how `analyzemodelnew()` identifies assignment, LHS, dependencies, lags, and leads
- where the model instance stores graph/order metadata
- which method handles topological ordering or simultaneous blocks in your version
